# Notebook 1 · Decomposition: trend & seasonality

Companion to lectures [4](https://jiangyou2025.github.io/kun/course/04/) and [5](https://jiangyou2025.github.io/kun/course/05/).

A real time series usually stacks three things at once: a **trend** (long-term drift), a
**seasonality / periodicity** (a pattern that repeats on a fixed cycle), and **noise** (random
wiggle). This notebook uses **synthetic data** (nothing to download) to pull them apart:

1. generate and plot a `trend + weekly + yearly + noise` series;
2. read trend and stationarity off the rolling mean / rolling std;
3. hand-code an **additive decomposition** `observed = trend + seasonal + residual`;
4. find the period with the autocorrelation function (ACF).

> Requires: `numpy`, `pandas`, `matplotlib`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)
n = 730                                   # two years of daily data
t = np.arange(n)

trend  = 0.05 * t                         # linear trend
weekly = 4  * np.sin(2*np.pi*t/7)         # weekly seasonality (period 7)
yearly = 10 * np.sin(2*np.pi*t/365.25)    # yearly seasonality (period 365)
noise  = np.random.normal(0, 2, n)        # noise
y = 20 + trend + weekly + yearly + noise

idx = pd.date_range('2022-01-01', periods=n, freq='D')
s = pd.Series(y, index=idx, name='value')
s.head()

## 1. Plot the raw series

Plot it first. The eye already picks out an upward drift (trend), regular bumps (seasonality)
and a fine jitter (noise).

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(s.index, s.values, lw=1)
plt.title('Synthetic daily series: trend + weekly + yearly + noise')
plt.xlabel('date'); plt.ylabel('value')
plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()

## 2. Rolling mean & rolling std

The **rolling mean** averages over a sliding window, smoothing the jitter so the **trend**
shows through. The **rolling std** tracks whether the spread changes over time (stationarity).
Here the trend makes the rolling mean drift up, while the rolling std stays roughly flat — the
variance is stable.

In [ ]:
roll_mean = s.rolling(30).mean()
roll_std  = s.rolling(30).std()

plt.figure(figsize=(11, 4))
plt.plot(s.index, s.values, alpha=.35, label='series')
plt.plot(roll_mean.index, roll_mean, label='30d rolling mean')
plt.plot(roll_std.index,  roll_std,  label='30d rolling std')
plt.legend(); plt.title('Rolling mean & std')
plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()

## 3. Additive decomposition (by hand)

The additive model assumes:

$$\text{observed} = \text{trend} + \text{seasonal} + \text{residual}$$

We estimate the pieces one at a time (using period 7 to demo the weekly season):

1. **trend** = centred moving average (window = period), which smooths out the season;
2. **detrended** = observed − trend;
3. **seasonal** = average the detrended series by its phase in the cycle (same weekday averaged);
4. **residual** = observed − trend − seasonal.

> In production reach for `statsmodels.tsa.seasonal.STL`; doing it by hand here shows what each
> step actually does.

In [ ]:
period = 7

# 1) trend: centred moving average
trend_est = s.rolling(period, center=True).mean()

# 2) detrend
detrended = s - trend_est

# 3) seasonal: average the detrended values by phase (day-of-week)
season_ix   = np.arange(len(s)) % period
season_mean = pd.Series(detrended.values, index=season_ix).groupby(level=0).mean()
seasonal    = pd.Series(season_mean.reindex(season_ix).values, index=s.index)

# 4) residual
residual = s - trend_est - seasonal

fig, ax = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
ax[0].plot(s.index, s);         ax[0].set_ylabel('observed')
ax[1].plot(s.index, trend_est); ax[1].set_ylabel('trend')
ax[2].plot(s.index, seasonal);  ax[2].set_ylabel('seasonal(7)')
ax[3].plot(s.index, residual);  ax[3].set_ylabel('residual')
ax[0].set_title('Additive decomposition: observed = trend + seasonal + residual')
for a in ax: a.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

## 4. Autocorrelation function (ACF)

**Autocorrelation** measures how similar the series is to a delayed copy of itself. If a period
exists, the ACF spikes at **lags that are integer multiples of the period** — the fingerprint of
periodicity.

In [ ]:
def acf(x, nlags=40):
    """Hand-coded autocorrelation for lag = 0..nlags."""
    x = np.asarray(x, float) - np.mean(x)
    var = np.dot(x, x)
    return np.array([np.dot(x[:len(x)-k], x[k:]) / var for k in range(nlags + 1)])

a = acf(s.values, 40)
plt.figure(figsize=(11, 4))
plt.stem(range(len(a)), a)
plt.axhline(0, color='k', lw=.8)
plt.title('Autocorrelation (ACF): note the spikes at lags 7, 14, 21 ...')
plt.xlabel('lag'); plt.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

## Summary

- A series is **trend + seasonality + noise**; the goal is to capture the first two and **not**
  fit the noise.
- Rolling statistics, decomposition and the ACF are your first tools for judging trend,
  seasonality and stationarity.
- The period you read off the ACF (here 7) becomes a feature or a baseline in the next step.
- Next: [Notebook 2](02_linear_from_scratch.ipynb) — write a linear forecaster from scratch.
